# 当前去重实验的误差重新分布分析

使用本项目 AusGrid2013 的 360 节点、GEF2017 的 10 节点结果。
AusGrid2013 覆盖 1 步，GEF2017 覆盖 1/3/6 步；两套数据都包括 FHF、HFF、HFFPS。
每种协调方法只与同一训练方法的 base 比较。尺度来自对应的已保存训练目标，
按当前语义层级平均，使用日历日区块 bootstrap 计算 95% 区间。

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ROOT = PROJECT_ROOT
import json
import pandas as pd
from IPython.display import Image, display
from PPHF_modules.results_check.error_analysis.error_redistribution_analysis import run_all

In [ ]:
OUTPUT_DIR = ROOT / 'Results_check' / 'deduplication_results' / 'error_analysis'
N_BOOTSTRAP = 2000
RECOMPUTE = False
manifest = run_all(root=ROOT, output_root=OUTPUT_DIR,
                   n_bootstrap=N_BOOTSTRAP, force=RECOMPUTE)
display(manifest)

## 分层效果与原指标复核

Delta 为协调后减 base，负值表示误差下降。
MASE/RMSSE 先逐节点计算，再在语义层级内等权平均；复核表与当前逐节点指标聚合结果比较。

In [ ]:
for dataset in ['ausgrid2013','gef2017']:
    folder = OUTPUT_DIR / dataset
    print(dataset)
    display(pd.read_csv(folder / 'level_effects_bootstrap.csv'))
    checks = pd.read_csv(folder / 'reproduction_check.csv')
    display(checks.groupby('method')[['abs_diff_MASE','abs_diff_RMSSE']].max())
    assert checks[['abs_diff_MASE','abs_diff_RMSSE']].to_numpy().max() <= 1e-6

## 底层节点改善比例与大误差观测

In [ ]:
for dataset in ['ausgrid2013','gef2017']:
    folder = OUTPUT_DIR / dataset
    print(dataset)
    display(pd.read_csv(folder / 'appendix_node_improvement.csv'))
    display(pd.read_csv(folder / 'largest_error_decile.csv'))
    display(Image(filename=str(folder / 'error_redistribution_heatmap.png')))

## 结果解释

误差十分位组由 base 误差定义，分组模式只作描述性解释。
bootstrap 区间描述当前保存实验内的变化，不等同于跨训练种子的稳定性结论。

In [ ]:
for dataset in ['ausgrid2013','gef2017']:
    records = json.loads((OUTPUT_DIR / dataset / 'validation.json').read_text())
    display(pd.DataFrame(records))
assert len(manifest) == 12
print('All outputs:', OUTPUT_DIR)
print('Source fingerprints:', OUTPUT_DIR / 'source_manifest.json')